# Day 2 — Data Understanding

## Dataset selection

The repository did not contain a raw e-commerce dataset at the start of Day 2. This notebook uses the **UCI Online Retail** dataset (UCI Machine Learning Repository, dataset ID 352) as the project's real source dataset. It contains transactional records for a UK-based non-store online retailer from 01/12/2010 through 09/12/2011. The source provides 541,909 instances and the transaction fields required for sales analysis and RFM/customer segmentation, including `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`.

The raw 22.6 MB workbook is intentionally **not committed** to GitHub. The notebook retrieves the dataset through `ucimlrepo` when executed, keeping the repository lightweight and reproducible.


In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

online_retail = fetch_ucirepo(id=352)
df = online_retail.data.features.copy()

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())


In [ ]:
# Column names and data types
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "unique": df.nunique(dropna=True)}))


In [ ]:
# Missing values and duplicate records
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values:")
display(missing[missing > 0].to_frame("missing_count"))
print(f"Duplicate rows: {df.duplicated().sum():,}")


In [ ]:
# Separate columns by broad analytical type
numeric_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(include=["object", "category"]).columns.tolist()
date_candidates = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]

print("Numeric columns:", numeric_columns)
print("Categorical/text columns:", categorical_columns)
print("Date/time candidates:", date_candidates)


In [ ]:
# Basic cardinality and value inspection for key categorical fields
for column in ["InvoiceNo", "StockCode", "CustomerID", "Country"]:
    if column in df.columns:
        print(f"{column}: {df[column].nunique(dropna=True):,} unique non-null values")

if "Country" in df.columns:
    display(df["Country"].value_counts().head(10).to_frame("row_count"))


In [ ]:
# Date parsing check without modifying the source dataframe
if "InvoiceDate" in df.columns:
    parsed_dates = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    print("Unparseable InvoiceDate values:", parsed_dates.isna().sum())
    print("Minimum InvoiceDate:", parsed_dates.min())
    print("Maximum InvoiceDate:", parsed_dates.max())


## Initial observations to validate when the notebook is executed

- Confirm the row/column count against the UCI source metadata.
- Inspect missingness, especially customer identifiers, before any customer-level analysis.
- Treat cancellation invoices and non-positive quantities/prices as data-quality topics for **Day 3**, not as silently removed records today.
- Keep `InvoiceDate` as a proper datetime field for later time-series and RFM analysis.
- Do not calculate revenue, RFM metrics, or customer segments until the cleaning rules are documented in Day 3.
